# Skin Cancer Detection: Kaggle eğitim koşusu

Bu notebook Mac'ten `kaggle kernels push -p kaggle` ile gönderilip Kaggle'da arka planda çalışır
(tarayıcı açık kalmak zorunda değil). Colab notebook'undan farkları:
- Veri Kaggle'daki HAM10000 veri setinden doğrudan bağlanır; API key gerekmez.
- Kod `/tmp`'ye klonlanır, böylece Kaggle çıktısına sadece sonuçlar (`/kaggle/working/models`) girer.
- Sonuçlar Mac'ten `kaggle kernels output yunusemreozkan/skin-cancer-train` ile çekilir.

Hangi modelin eğitileceği aşağıdaki ayar hücresinde.

In [ ]:
# Bu koşunun ayarlari: mimari degisince sadece burasi degisiyor.
ARCHITECTURE = 'convnext_tiny'
EXTRA_TRAIN_ARGS = '--image-size 384'  # ornek: '--batch-size 16'

In [ ]:
!nvidia-smi

In [ ]:
%cd /tmp
!rm -rf skin-cancer-detection
!git clone --depth 1 https://github.com/yunus345/skin-cancer-detection.git
%cd /tmp/skin-cancer-detection
!git log --oneline -1

In [ ]:
# Kaggle'da torch/torchvision CUDA'li kurulu geliyor; sadece augmentation kutuphanesini
# yerelde test ettigimiz surume sabitliyoruz.
!pip install -q albumentations==2.0.8

In [ ]:
import glob, os

# Kaggle'in veri setini bagladigi klasoru metadata dosyasindan buluyoruz (yol zamanla degisebiliyor).
matches = glob.glob('/kaggle/input/**/HAM10000_metadata.csv', recursive=True)
assert matches, 'HAM10000_metadata.csv bulunamadi: veri seti bagli mi?'
DATA_ROOT = os.path.dirname(matches[0])
print('Veri klasoru:', DATA_ROOT)

!python -m src.data.build_manifest --data-root {DATA_ROOT} --out data/processed/manifest.csv
!python -m src.data.split_manifest --manifest data/processed/manifest.csv --out data/processed/manifest.csv
!python -m src.data.validate_manifest --manifest data/processed/manifest.csv

In [ ]:
# Kaggle GPU makinelerinde 4 CPU cekirdegi var; veri yuklemeyi 4 yardimci surece dagitiyoruz.
!python -m src.training.train \
  --feature-extract-epochs 5 \
  --finetune-epochs 20 \
  --patience 6 \
  --architecture {ARCHITECTURE} \
  --num-workers 4 \
  --device auto {EXTRA_TRAIN_ARGS}

In [ ]:
!python -m src.evaluation.evaluate --device auto
!python -m src.evaluation.plot_history
!python -m src.evaluation.plot_confusion_matrix

In [ ]:
# Sadece /kaggle/working altindakiler Kaggle ciktisi olarak saklaniyor.
!rm -rf /kaggle/working/models
!cp -r models /kaggle/working/models
!ls -la /kaggle/working/models